# 🔵 Notebook 04: Clustering (Pengelompokan Data)
## Analitika Data Keuangan Sektor Publik

**Tujuan Pembelajaran:**
1. Memahami perbedaan supervised vs unsupervised learning
2. Menerapkan algoritma K-Means clustering
3. Menggunakan Elbow Method dan Silhouette Score untuk menentukan K optimal
4. Memvisualisasikan dan menginterpretasikan hasil klaster
5. Memberi profil / karakterisasi setiap klaster

---
> **Problem:** Kelompokkan pemerintah daerah berdasarkan **profil keuangan fiskalnya** tanpa menggunakan label Predikat yang sudah ada — temukan pola pengelompokan alami dalam data!

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print('Library berhasil dimuat ✅')

## 📂 2. Persiapan Data

In [ ]:
# Muat dan bersihkan data
df = pd.read_csv('../../Dataset/02-EDA/keuangan_pemda.csv')
df = df[df['Anggaran_APBD'] > 0].copy()
df.dropna(subset=['Anggaran_APBD', 'Realisasi_APBD', 'PAD', 'Dana_Transfer'], inplace=True)

# Feature Engineering — Rasio (lebih informatif dari nilai absolut)
df['Pct_Realisasi']          = df['Realisasi_APBD'] / df['Anggaran_APBD'] * 100
df['Rasio_Kemandirian']      = df['PAD'] / df['Anggaran_APBD'] * 100
df['Rasio_Transfer']         = df['Dana_Transfer'] / df['Anggaran_APBD'] * 100
df['Rasio_Belanja_Pegawai']  = df['Belanja_Pegawai'] / df['Realisasi_APBD'] * 100
df['Rasio_Belanja_Modal']    = df['Belanja_Modal'] / df['Realisasi_APBD'] * 100

# Feature untuk clustering
cluster_features = ['Pct_Realisasi', 'Rasio_Kemandirian', 'Rasio_Transfer',
                    'Rasio_Belanja_Pegawai', 'Rasio_Belanja_Modal']

X = df[cluster_features].replace([np.inf, -np.inf], np.nan).dropna()
df_cluster = df.loc[X.index].copy()

print(f'Data siap untuk clustering: {X.shape[0]} PEMDA × {X.shape[1]} fitur')
print(f'\nFitur yang digunakan:')
for f in cluster_features:
    print(f'  - {f}')

In [ ]:
# Normalisasi data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Setelah StandardScaler (mean=0, std=1):')
pd.DataFrame(X_scaled, columns=cluster_features).describe().round(2)

## 📐 3. Menentukan Jumlah Klaster Optimal

In [ ]:
# Elbow Method + Silhouette Score
k_range = range(2, 10)
inertias, silhouette_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(k_range, inertias, 'b-o', markersize=8, linewidth=2)
axes[0].set_xlabel('Jumlah Klaster (K)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].set_title('Elbow Method\nCari titik "siku"', fontweight='bold')
axes[0].set_xticks(list(k_range))

# Silhouette
best_k = k_range[np.argmax(silhouette_scores)]
colors = ['red' if k == best_k else 'steelblue' for k in k_range]
axes[1].bar(k_range, silhouette_scores, color=colors, edgecolor='white')
axes[1].set_xlabel('Jumlah Klaster (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title(f'Silhouette Score\nK terbaik = {best_k} (merah)', fontweight='bold')
axes[1].set_xticks(list(k_range))

for k, sc in zip(k_range, silhouette_scores):
    axes[1].text(k, sc + 0.005, f'{sc:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()
print(f'\nK optimal berdasarkan Silhouette Score: {best_k}')
print(f'Silhouette Score: {max(silhouette_scores):.4f}')

## 🎯 4. Melatih K-Means

In [ ]:
# Jalankan K-Means dengan K terbaik
K = best_k
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(X_scaled)

df_cluster['Klaster'] = kmeans.labels_

print(f'K-Means dengan K={K}')
print(f'Inertia: {kmeans.inertia_:.2f}')
print(f'Silhouette Score: {silhouette_score(X_scaled, kmeans.labels_):.4f}')
print()
print('Distribusi anggota per klaster:')
print(df_cluster['Klaster'].value_counts().sort_index())

## 📊 5. Visualisasi Klaster

In [ ]:
# Reduksi ke 2D dengan PCA untuk visualisasi
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter klaster (tanpa label)
colors_cluster = plt.cm.Set1(np.linspace(0, 0.9, K))
for k in range(K):
    mask = kmeans.labels_ == k
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[colors_cluster[k]], label=f'Klaster {k}', alpha=0.7, s=60)

# Centroid di PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                marker='*', s=300, c='black', label='Centroid', zorder=5)
axes[0].set_title(f'K-Means Clustering (K={K})\nDireduksi ke 2D via PCA', fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].legend()

# Scatter dengan Predikat asli (untuk perbandingan)
predikat_colors = {'Kurang': '#e74c3c', 'Cukup': '#f39c12', 'Baik': '#2ecc71', 'Sangat Baik': '#27ae60'}
for predikat, color in predikat_colors.items():
    mask_p = df_cluster['Predikat'] == predikat
    if mask_p.sum() > 0:
        idx = df_cluster.index[mask_p]
        pos = [i for i, ix in enumerate(df_cluster.index) if ix in idx]
        if pos:
            axes[1].scatter(X_pca[pos, 0], X_pca[pos, 1],
                            c=color, label=predikat, alpha=0.7, s=60)
axes[1].set_title('Label Predikat Asli (untuk perbandingan)\nApakah klaster sesuai label?', fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[1].legend(title='Predikat')

plt.tight_layout()
plt.show()

## 🏷️ 6. Profil Karakteristik Setiap Klaster

In [ ]:
# Analisis karakteristik per klaster
profil = df_cluster.groupby('Klaster')[cluster_features + ['Predikat']].agg(
    lambda x: x.mean() if x.dtype != 'object' else x.value_counts().index[0]
).round(2)

print('PROFIL RATA-RATA PER KLASTER')
print('=' * 70)
print(profil.to_string())
print()

# Distribusi Predikat per klaster
print('DISTRIBUSI PREDIKAT DALAM SETIAP KLASTER')
cross_tab = pd.crosstab(df_cluster['Klaster'], df_cluster['Predikat'])
print(cross_tab)

In [ ]:
# Radar Chart — Profil klaster
from matplotlib.patches import FancyArrowPatch

features_plot = cluster_features
N = len(features_plot)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # Close the plot

fig, ax = plt.subplots(1, 1, figsize=(8, 8), subplot_kw=dict(projection='polar'))

centroid_df = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=cluster_features
)

# Normalize untuk radar
norm_centers = (centroid_df - centroid_df.min()) / (centroid_df.max() - centroid_df.min() + 1e-9)

cluster_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for k in range(K):
    values = norm_centers.iloc[k].values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Klaster {k}', color=cluster_colors[k])
    ax.fill(angles, values, alpha=0.15, color=cluster_colors[k])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace('_', '\n').replace('Rasio ', ''), 'feat1'], fontsize=9)
ax.set_xticklabels([f.replace('Rasio_', '').replace('_', '\n') for f in features_plot], fontsize=9)
ax.set_title(f'Radar Chart Profil Klaster (K={K})', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot per klaster untuk feature utama
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, feature in zip(axes, ['Pct_Realisasi', 'Rasio_Kemandirian', 'Rasio_Belanja_Pegawai']):
    data_to_plot = [df_cluster[df_cluster['Klaster'] == k][feature].dropna().values
                    for k in range(K)]
    bp = ax.boxplot(data_to_plot, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], cluster_colors[:K]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(feature.replace('_', ' '), fontweight='bold')
    ax.set_xticklabels([f'K-{k}' for k in range(K)])
    ax.set_ylabel('Nilai (%)')

plt.suptitle('Distribusi Feature per Klaster', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Interpretasi klaster
print('=' * 60)
print('INTERPRETASI KLASTER')
print('=' * 60)
for k in range(K):
    subset = df_cluster[df_cluster['Klaster'] == k]
    pct = subset['Pct_Realisasi'].mean()
    mandiri = subset['Rasio_Kemandirian'].mean()
    predikat_dominant = subset['Predikat'].value_counts().index[0] if len(subset) > 0 else 'N/A'
    
    print(f'\nKlaster {k} ({len(subset)} PEMDA):')
    print(f'  Avg % Realisasi    : {pct:.1f}%')
    print(f'  Avg Rasio Mandiri  : {mandiri:.1f}%')
    print(f'  Predikat Dominan   : {predikat_dominant}')
    print(f'  5 PEMDA pertama    : {list(subset["Kode_Pemda"].head())}')

## 📝 7. Latihan

1. Coba ubah `K=4` secara manual. Apakah hasilnya lebih bermakna?
2. Tambahkan fitur `Rasio_Belanja_Modal` dan ulangi clustering. Apakah klaster berubah?
3. Bandingkan K-Means dengan **Agglomerative Clustering** dari `sklearn.cluster`.
4. Buat **heatmap** yang menunjukkan rata-rata setiap fitur per klaster.
5. Berikan **nama** yang bermakna untuk setiap klaster berdasarkan profilnya!

In [ ]:
# ✏️ Ruang Eksplorasi Mandiri
# Heatmap profil klaster
fig, ax = plt.subplots(figsize=(10, 4))
heatmap_data = df_cluster.groupby('Klaster')[cluster_features].mean().round(1)
sns.heatmap(heatmap_data.T, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Nilai'})
ax.set_title('Heatmap: Profil Rata-Rata per Klaster', fontweight='bold')
ax.set_xlabel('Klaster')
plt.tight_layout()
plt.show()